# Example queries: `applied_helpers` (comstock_oedi_agg)

Auto-generated from `tests/query_snapshots/applied_helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_agg_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_agg_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_agg_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `applied_buildings_all_of_1_2`

get_applied_buildings(all_of=[1, 2]) — buildings where both upgrades 1 and 2 applied successfully. Pins the all_of-only branch SQL: WHERE upgrade IN (1,2) AND applicability GROUP BY <md_keys> HAVING count(distinct(upgrade))=2. Same shape as the prior `applied_in=[1,2]` form.


In [3]:
result = bsq.get_applied_buildings(all_of=[1, 2])
result.head() if hasattr(result, 'head') else result


,bldg_id,county,state
0,142568,G5400850,WV
1,91841,G5000070,VT
2,39083,G0600070,CA
3,48290,G0801070,CO
4,68197,G1302330,GA


## `applied_buildings_any_of_1_2`

get_applied_buildings(any_of=[1, 2]) — buildings where at least one of upgrades 1 or 2 applied successfully. Pins the any_of-only branch SQL: WHERE upgrade IN (1,2) AND applicability GROUP BY <md_keys> (no HAVING — GROUP BY dedupes).


In [4]:
result = bsq.get_applied_buildings(any_of=[1, 2])
result.head() if hasattr(result, 'head') else result


,bldg_id,county,state
0,7070,G0202400,AK
1,6310,G0201850,AK
2,6909,G0201850,AK
3,6796,G0201100,AK
4,5544,G0201700,AK


## `applied_buildings_all_of_1_any_of_2_3`

get_applied_buildings(all_of=[1], any_of=[2, 3]) — buildings where upgrade 1 applied AND at least one of upgrades 2 or 3 also applied. Pins the both-lists branch SQL with the CASE-WHEN HAVING form.


In [5]:
result = bsq.get_applied_buildings(all_of=[1], any_of=[2, 3])
result.head() if hasattr(result, 'head') else result


,bldg_id,county,state
0,93693,G2400250,MD
1,150446,G3600050,NY
2,114844,G3600290,NY
3,142074,G5400150,WV
4,74656,G1600690,ID


## `query_with_applied_buildings_filter`

Annual baseline restricted via `restrict=[get_applied_buildings_filter(all_of=[1,2]), (state, [CO])]`. Confirms the filter-tuple composes cleanly into the standard restrict pipeline (composite-key tuple-IN on multi-key schemas).


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    upgrade_id='0',
    restrict=[{'_applied_filter': {'all_of': [1, 2]}}, ('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


ValidationError: 2 validation errors for Query
restrict.0.tuple[union[is-instance[Label],is-instance[Column],is-instance[ColumnElement],str,MappedColumn], union[str,int,bool,json-or-python[json=list[union[int,str,bool]],python=chain[is-instance[Sequence],function-wrap[sequence_validator()]]],is-instance[SelectBase],is-instance[Subquery]]]
  Input should be a valid tuple [type=tuple_type, input_value={'_applied_filter': {'all_of': [1, 2]}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/tuple_type
restrict.0.`tuple[tuple[union[is-instance[Label],is-instance[Column],is-instance[ColumnElement]], ...], union[is-instance[SelectBase],is-instance[Subquery],json-or-python[json=list[tuple[union[str,int,bool], ...]],python=chain[is-instance[Sequence],function-wrap[sequence_validator()]]]]]`
  Input should be a valid tuple [type=tuple_type, input_value={'_applied_filter': {'all_of': [1, 2]}}, input_type=dict]
    For further information visit https://errors.